In [ ]:
# Task 05 check: did the two private datasets arrive on Kaggle intact? Prints a summary and saves check_result.json
import json, os, subprocess, sys
from pathlib import Path

inp = Path("/kaggle/input")
print("Top of /kaggle/input:")
for p in sorted(inp.rglob("*")):
    depth = len(p.relative_to(inp).parts)
    if depth <= 4 and (p.is_dir() or depth <= 3) and p.parent.name not in ("png1024", "masks1024"):
        print("  " * depth + p.name + ("/" if p.is_dir() else f"  ({p.stat().st_size} bytes)"))

# find the folders wherever Kaggle mounted them
data_root = next(p.parent for p in inp.rglob("png1024") if p.is_dir())
code_root = next(p.parent.parent for p in inp.rglob("src/config.py"))
print("\ndata dataset at:", data_root)
print("code dataset at:", code_root)

pngs = sorted((data_root / "png1024").glob("*.png"))
masks = sorted((data_root / "masks1024").glob("*.png"))
manifest = json.loads((data_root / "manifest.json").read_text())
result = {"data_root": str(data_root), "code_root": str(code_root),
          "n_png": len(pngs), "n_masks": len(masks),
          "png_bytes": sum(p.stat().st_size for p in pngs), "mask_bytes": sum(p.stat().st_size for p in masks),
          "index_files": sorted(p.name for p in (data_root / "index").glob("*.csv")),
          "split_files": sorted(p.name for p in (data_root / "splits").glob("*.csv")),
          "manifest": manifest}
print(f"\nPNGs: {len(pngs)} (manifest says {manifest['n_png']})")
print(f"masks: {len(masks)} (manifest says {manifest['n_masks']})")

import pandas as pd
pairs = pd.read_csv(data_root / "index" / "pairs.csv")
s1 = pd.read_csv(data_root / "splits" / "stage1.csv")
s2 = pd.read_csv(data_root / "splits" / "stage2.csv")
e0 = pd.read_csv(data_root / "splits" / "stage2_e0_imagewise.csv")
names = {p.name for p in pngs}
mask_names = {p.name for p in masks}
result.update(
    pairs_rows=len(pairs), stage1_breasts=len(s1), stage2_patients=len(s2), e0_images=len(e0),
    stage1_per_split=s1["split"].value_counts().to_dict(), stage2_per_split=s2["split"].value_counts().to_dict(),
    stage2_has_box_only=int(s2["has_box_only"].sum()),
    pngs_named_in_pairs_but_missing=int((~pd.concat([pairs["png_cc"], pairs["png_mlo"]]).isin(names)).sum()),
    masks_named_in_pairs_but_missing=int((~pairs["mask"].dropna().isin(mask_names)).sum()),
    patients_in_two_stage1_splits=int((s1.groupby("patient_id")["split"].nunique() > 1).sum()))
print("pairs rows:", len(pairs), "| stage1:", result["stage1_per_split"], "| stage2:", result["stage2_per_split"])
print("PNGs named in pairs.csv but missing:", result["pngs_named_in_pairs_but_missing"],
      "| masks missing:", result["masks_named_in_pairs_but_missing"])

# open one Stage 2 breast: CC, MLO and mask
from PIL import Image
r = s2.iloc[0]
cc = Image.open(data_root / "png1024" / r["png_cc"]); mlo = Image.open(data_root / "png1024" / r["png_mlo"])
mask = Image.open(data_root / "masks1024" / r["mask"])
result["example"] = {"patient": r["patient_id"], "subtype": r["subtype"], "cc_size": cc.size, "mlo_size": mlo.size,
                     "mask_size": mask.size, "mode": cc.mode, "mask_same_size_as_mlo": mask.size == mlo.size}
print("example:", result["example"])

# our own code imports from the code dataset (no GitHub token needed)
sys.path.insert(0, str(code_root))
import src.config
result["src_import_ok"] = hasattr(src.config, "load_config")
result["requirements_train"] = (code_root / "requirements-train.txt").read_text().split("\n")[1:]
print("src.config imported:", result["src_import_ok"])

# GPU
import torch
result["torch"] = torch.__version__
result["cuda_available"] = torch.cuda.is_available()
result["gpu_count"] = torch.cuda.device_count()
result["gpu_name"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
result["python"] = sys.version.split()[0]
for pkg in ["monai", "timm", "radiomics", "SimpleITK", "pytorch_grad_cam", "shap", "sklearn", "cv2"]:
    try:
        m = __import__(pkg); result["pkg_" + pkg] = getattr(m, "__version__", "installed")
    except Exception as e:
        result["pkg_" + pkg] = "NOT INSTALLED"
print("torch", result["torch"], "| cuda:", result["cuda_available"], "| GPUs:", result["gpu_count"], "|", result["gpu_name"])
print({k: v for k, v in result.items() if k.startswith("pkg_")})

result["all_ok"] = bool(len(pngs) == manifest["n_png"] == 5202 and len(masks) == manifest["n_masks"] == 1385
                        and result["pngs_named_in_pairs_but_missing"] == 0
                        and result["masks_named_in_pairs_but_missing"] == 0
                        and result["patients_in_two_stage1_splits"] == 0 and result["src_import_ok"])
Path("/kaggle/working/check_result.json").write_text(json.dumps(result, indent=2, default=str))
print("\nALL OK" if result["all_ok"] else "\nSOMETHING IS WRONG", "-> check_result.json")